This file contains the data analysis and inferences regarding the dataset 
The main objectives are
1) to tell how deviated the avg data is from all the files individually 
2) if all the flex sensor readings fall in the same range (max deviation acceptabel is 10%)
3) decide onto the best practices which I can partake in regarding this dataset, for now I plan to use xgboost or random forest.
4) also to check if there are some missing values in my data or not if yes manage it
5) it can also be used to do further encodings and labellings for future updates

Data ingestion
Getting the average files regarding the folders and comparing them simultaneously
Then checking it against the combined average too
and checking individual average to the combined averages

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
# Paths (change only if needed)
AVG_CSV_PATH = "dataset/all_mean_files/combined_alphabet_mean_dataset.csv"
RAW_BASE_DIR = "dataset"

# Flex sensors only
FLEX_COLS = ["flex_1", "flex_2", "flex_3", "flex_4", "flex_5"]


In [3]:
avg_df = pd.read_csv(AVG_CSV_PATH)
avg_df.head()


,timestamp,user_id,flex_1,flex_2,flex_3,flex_4,flex_5,Qw,Qx,Qy,...,ACCx,ACCy,ACCz,ACCx_body,ACCy_body,ACCz_body,ACCx_world,ACCy_world,ACCz_world,label
0,1.616133e+09,1,21.596667,58.080667,72.341333,74.798667,57.697333,0.790068,0.108744,-0.602366,...,9.190709,1.481503,2.361693,-0.163263,-0.050611,-0.082186,0.041760,-0.017065,-0.183465,A
1,1.616133e+09,1,60.256667,2.076667,-1.661333,-8.564667,-7.944667,0.694574,-0.000514,-0.718142,...,9.474952,0.265658,-0.346571,-0.289519,-0.031147,-0.023955,0.031220,-0.023577,-0.290355,B
2,1.616133e+09,1,49.180000,46.440667,61.716667,60.138000,41.744000,0.811679,0.146621,-0.561436,...,8.780222,2.775029,3.126730,-0.043687,-0.026759,-0.059952,0.041930,-0.003945,-0.065854,C
3,1.616133e+09,1,57.812667,-6.532000,66.176000,71.766667,53.604000,0.787819,0.190831,-0.568787,...,9.038972,2.373083,2.503404,-0.092172,-0.031600,-0.056248,0.031084,-0.003360,-0.109283,D
4,1.616133e+09,1,70.804000,51.616667,76.712667,71.656667,51.017333,0.779161,0.064974,-0.622582,...,9.282545,1.040415,2.022744,-0.207554,-0.045618,-0.082969,0.036673,-0.019088,-0.224731,E


In [4]:
def load_raw_flex_data(letter):
    letter = letter.lower()
    all_data = []

    for folder in os.listdir(RAW_BASE_DIR):
        if folder.isdigit():
            csv_path = os.path.join(
                RAW_BASE_DIR, folder, "alphabets", f"{letter}.csv"
            )

            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)
                all_data.append(df[FLEX_COLS])

    if all_data:
        return pd.concat(all_data, ignore_index=True)
    else:
        return None


In [5]:
'''letter = "A"

avg_row = avg_df[avg_df["label"] == letter].iloc[0]
raw_df = load_raw_flex_data(letter)

plt.figure()

for col in FLEX_COLS:
    plt.plot(raw_df[col].values, alpha=0.3)
    plt.hlines(
        avg_row[col],
        xmin=0,
        xmax=len(raw_df),
        linestyles="dashed"
    )

plt.title(f"Flex Sensor Validation for Alphabet '{letter}'")
plt.xlabel("Sample Index")
plt.ylabel("Flex Sensor Value")
plt.show()'''


'letter = "A"\n\navg_row = avg_df[avg_df["label"] == letter].iloc[0]\nraw_df = load_raw_flex_data(letter)\n\nplt.figure()\n\nfor col in FLEX_COLS:\n    plt.plot(raw_df[col].values, alpha=0.3)\n    plt.hlines(\n        avg_row[col],\n        xmin=0,\n        xmax=len(raw_df),\n        linestyles="dashed"\n    )\n\nplt.title(f"Flex Sensor Validation for Alphabet \'{letter}\'")\nplt.xlabel("Sample Index")\nplt.ylabel("Flex Sensor Value")\nplt.show()'

In [6]:
'''for _, avg_row in avg_df.iterrows():
    letter = avg_row["label"]

    raw_df = load_raw_flex_data(letter)
    if raw_df is None:
        continue

    plt.figure()

    for col in FLEX_COLS:
        plt.plot(raw_df[col].values, alpha=0.3)
        plt.hlines(
            avg_row[col],
            xmin=0,
            xmax=len(raw_df),
            linestyles="dashed"
        )

    plt.title(f"Flex Sensor Validation for Alphabet '{letter}'")
    plt.xlabel("Samp le Index")
    plt.ylabel("Flex Sensor Value")
    plt.show()'''


'for _, avg_row in avg_df.iterrows():\n    letter = avg_row["label"]\n\n    raw_df = load_raw_flex_data(letter)\n    if raw_df is None:\n        continue\n\n    plt.figure()\n\n    for col in FLEX_COLS:\n        plt.plot(raw_df[col].values, alpha=0.3)\n        plt.hlines(\n            avg_row[col],\n            xmin=0,\n            xmax=len(raw_df),\n            linestyles="dashed"\n        )\n\n    plt.title(f"Flex Sensor Validation for Alphabet \'{letter}\'")\n    plt.xlabel("Samp le Index")\n    plt.ylabel("Flex Sensor Value")\n    plt.show()'

In [7]:
print(os.listdir(RAW_BASE_DIR))

['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '018', '019', '020', '021', '022', '023', '024', '025', 'all_mean_files', 'average', 'master_alphabets_test.csv', 'master_alphabets_train.csv']


In [8]:
BASE_DIR = "dataset"   # contains 001–025 folders
FLEX_COLS = ["flex_1", "flex_2", "flex_3", "flex_4", "flex_5"]

In [9]:
def plot_file_vs_own_mean(folder_id, letter):
    csv_path = os.path.join(
        BASE_DIR, folder_id, "alphabets", f"{letter.lower()}.csv"
    )

    if not os.path.exists(csv_path):
        return

    df = pd.read_csv(csv_path)

    # Compute mean from THIS file only
    mean_vals = df[FLEX_COLS].mean()

    plt.figure()

    for col in FLEX_COLS:
        plt.plot(df[col].values, alpha=0.4)
        plt.hlines(
            mean_vals[col],
            xmin=0,
            xmax=len(df),
            linestyles="dashed"
        )

    plt.title(f"Folder {folder_id} — Letter '{letter}' (Raw vs Own Mean)")
    plt.xlabel("Sample Index (Time Step)")
    plt.ylabel("Flex Sensor Value")
    plt.show()


In [10]:
plot_file_vs_own_mean("001", "A")


In [ ]:
'''for folder in sorted(os.listdir(BASE_DIR)):
    if folder.isdigit():
        for letter in string.ascii_uppercase:
            plot_file_vs_own_mean(folder, letter)'''


NameError: name 'string' is not defined

Guess majority of values do lie in the mean range cool stuff 
There are some outliers (decide on the plan of action)
Lets make a model first from the average value data and then lets start fine tuning it. 

In [ ]:

# Load datasets
train_df = pd.read_csv("dataset/master_alphabets_train.csv")
test_df = pd.read_csv("dataset/master_alphabets_test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

<>:1: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:1: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
C:\Users\pieush\AppData\Local\Temp\ipykernel_13068\775743816.py:1: SyntaxWarning: invalid escape sequence '\m'
  Test_dir = "dataset\master_alphabets_test.csv"
C:\Users\pieush\AppData\Local\Temp\ipykernel_13068\775743816.py:2: SyntaxWarning: invalid escape sequence '\m'
  Train_dir = "dataset\master_alphabets_train.csv"


,timestamp,user_id,flex_1,flex_2,flex_3,flex_4,flex_5,Qw,Qx,Qy,...,ACCx,ACCy,ACCz,ACCx_body,ACCy_body,ACCz_body,ACCx_world,ACCy_world,ACCz_world,label
0,1.616133e+09,1.0,20.0,58.0,72.0,77.0,58.0,0.773499,0.087097,-0.627747,...,9.314306,1.379321,1.770508,-0.205762,0.082544,-0.155518,0.098096,0.124414,-0.218921,A
1,1.616133e+09,1.0,20.0,58.0,70.0,76.0,58.0,0.773438,0.086060,-0.627991,...,9.290380,1.372144,1.762134,-0.232080,0.086133,-0.162695,0.099292,0.130396,-0.245239,A
2,1.616133e+09,1.0,20.0,58.0,73.0,75.0,60.0,0.773376,0.084900,-0.628235,...,9.301147,1.341040,1.754956,-0.222510,0.066992,-0.167480,0.107666,0.111255,-0.239258,A
3,1.616133e+09,1.0,19.0,58.0,73.0,79.0,58.0,0.773254,0.083679,-0.628540,...,9.331055,1.271655,1.739404,-0.194995,0.011963,-0.179443,0.131592,0.055029,-0.222510,A
4,1.616133e+09,1.0,3.0,58.0,72.0,80.0,58.0,0.773193,0.082642,-0.628784,...,9.376513,1.195093,1.711890,-0.153125,-0.051440,-0.204565,0.171069,-0.008374,-0.194995,A
